# Notebook 1: Project Setup and Baseline
## SmartCache ML: Semantic Caching and Cost-Aware LLM Routing

### 1. Problem Overview
Language model inference is computationally expensive, has non-trivial latency (often hundreds of milliseconds to multiple seconds), and costs money per input/output token when using cloud APIs. In many production systems (e.g. customer support, enterprise internal search, FAQs, developer docs), users frequently ask identical or semantically equivalent questions.

Repeatedly forwarding identical or near-identical prompts to large language models wastes compute and budget. In this notebook, we establish our project baseline:
1. We inspect our benchmark dataset.
2. We establish a **No-Cache Baseline** where every query triggers full model inference.
3. We build an **Exact-Match Cache** and observe why it fails on realistic user traffic.
4. We formalize our evaluation metrics: latency (p50, p95), inference cost (USD), cache hit rate, and answer quality.

---
### Learning Objectives
By the end of this notebook, you will be able to:
- Formulate the cost and latency bottlenecks of un-cached LLM systems.
- Explain the mechanics of exact-match string caching and its limitations.
- Define reproducible evaluation metrics: mean, p50, and p95 latency, financial cost estimations, and token overlap quality.
- Articulate the baseline performance numbers for technical interview discussions.

In [ ]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from smartcache.config import settings
from smartcache.llm_provider import MockProvider
from smartcache.cache import ExactMatchCache
from smartcache.cost_tracker import CostTracker, RequestRecord
from smartcache.metrics import calculate_token_overlap_f1

print("Environment and SmartCache modules loaded successfully!")
print(f"Data raw path: {settings.data_raw_path}")

### 2. Loading and Inspecting the Benchmark Dataset
Let's load the benchmark queries generated for this project and examine class balances, categories, and paraphrase distributions.

In [ ]:
dataset_path = settings.data_raw_path / "benchmark_dataset.csv"
if not dataset_path.exists():
    raise FileNotFoundError(f"Dataset not found at {dataset_path}. Run generate_sample_data.py first.")

df = pd.read_csv(dataset_path)
print(f"Total queries: {len(df)}")
print(f"Unique paraphrase groups: {df['paraphrase_group'].nunique()}")
print("\nClass distribution:")
print(df['complexity_label'].value_counts())
print("\nCategory distribution:")
print(df['category'].value_counts())
df.head(6)

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
df['complexity_label'].value_counts().plot(kind='bar', color=['#2b5c8f', '#d95f02'])
plt.title("Query Complexity Distribution")
plt.ylabel("Count")

plt.subplot(1, 2, 2)
df['category'].value_counts().plot(kind='barh', color='#7570b3')
plt.title("Queries by Category")
plt.xlabel("Count")
plt.tight_layout()
plt.show()

### 3. Establishing the No-Cache Baseline
In the no-cache baseline, every incoming query is dispatched directly to our LLM provider.
We measure total latency, estimated token counts, and cost.

In [ ]:
# Instantiate MockProvider with deterministic answers
provider = MockProvider()
tracker_baseline = CostTracker()

# Simulate serving the entire query workload without caching (always using large model as baseline standard)
print("Running No-Cache Baseline over test queries...")
sample_queries = df.head(30)

for _, row in sample_queries.iterrows():
    t_start = time.perf_counter()
    resp = provider.generate(row['query_text'], model_type="large")
    latency = time.perf_counter() - t_start
    
    cost = tracker_baseline.calculate_cost(
        route="large",
        input_tokens=resp.input_tokens,
        output_tokens=resp.output_tokens,
        is_cache_hit=False
    )
    
    rec = RequestRecord(
        request_id=f"base_{row['query_id']}",
        query_text=row['query_text'],
        cache_hit=False,
        similarity_score=None,
        selected_route="large",
        is_mock=True,
        input_tokens=resp.input_tokens,
        output_tokens=resp.output_tokens,
        latency_seconds=latency,
        estimated_cost_usd=cost,
        generated_answer=resp.text,
        reference_answer=row['expected_answer']
    )
    tracker_baseline.record_request(rec)

summary_baseline = tracker_baseline.get_summary()
print("\n=== NO-CACHE BASELINE SUMMARY ===")
for k, v in summary_baseline.items():
    print(f"{k}: {v}")

### 4. Implementing and Testing an Exact-Match Cache
An exact-match cache keys on normalized string equality (`query.strip().lower()`).
Let's see what happens when canonical questions are cached, and users ask paraphrases!

In [ ]:
exact_cache = ExactMatchCache()
tracker_exact = CostTracker()

# Step 1: Pre-populate cache with canonical queries
canonicals = sample_queries[sample_queries['query_role'] == 'canonical']
for _, row in canonicals.iterrows():
    exact_cache.store(row['query_text'], row['expected_answer'], is_cacheable=row['is_cacheable'])

print(f"Pre-populated exact cache with {len(canonicals)} canonical questions.\n")

# Step 2: Now simulate incoming traffic of paraphrases
for _, row in sample_queries.iterrows():
    lookup = exact_cache.lookup(row['query_text'])
    
    if lookup.hit:
        rec = RequestRecord(
            request_id=f"exact_{row['query_id']}",
            query_text=row['query_text'],
            cache_hit=True,
            similarity_score=1.0,
            selected_route="cache",
            is_mock=False,
            input_tokens=0,
            output_tokens=0,
            latency_seconds=lookup.total_lookup_latency,
            estimated_cost_usd=0.0,
            generated_answer=lookup.response
        )
    else:
        # Cache miss: generate answer
        t_start = time.perf_counter()
        resp = provider.generate(row['query_text'], model_type="large")
        lat = time.perf_counter() - t_start
        cost = tracker_exact.calculate_cost("large", resp.input_tokens, resp.output_tokens, False)
        rec = RequestRecord(
            request_id=f"exact_{row['query_id']}",
            query_text=row['query_text'],
            cache_hit=False,
            similarity_score=None,
            selected_route="large",
            is_mock=True,
            input_tokens=resp.input_tokens,
            output_tokens=resp.output_tokens,
            latency_seconds=lat,
            estimated_cost_usd=cost,
            generated_answer=resp.text
        )
    tracker_exact.record_request(rec)

summary_exact = tracker_exact.get_summary()
print("=== EXACT-MATCH CACHE SUMMARY ===")
print(f"Total Requests: {summary_exact['total_requests']}")
print(f"Cache Hits: {summary_exact['cache_hits']}")
print(f"Cache Misses: {summary_exact['cache_misses']}")
print(f"Hit Rate: {summary_exact['cache_hit_rate'] * 100:.1f}%")
print(f"Total Cost: ${summary_exact['total_cost_usd']:.6f}")

### 5. Interpretation & Limitations of Exact-Match Caching
Notice that the exact-match cache only registers hits on verbatim matches. When a user asks:
- *"Can you tell me the capital city of France?"* instead of *"What is the capital of France?"*
The exact-match cache reports a complete **cache miss**, forwarding the duplicate request to the LLM. In typical real-world traffic, verbatim duplication is low (< 10%), while semantic duplication can reach 30%–60%.

---
### 6. Interview Concepts Checklist
Be prepared to explain these concepts in a Machine Learning Engineer interview:
1. **The Cost of Generative Inference**: Why LLM inference is orders of magnitude more expensive than embedding lookup or tabular inference (autoregressive token-by-token generation vs. forward pass).
2. **P50 vs P95 Latency**: Why mean latency hides tail latency spikes, and why caching specifically flattens tail latency.
3. **Exact vs Semantic Matching**: The failure modes of string hashing ($O(1)$ lookup with MD5/SHA256) when exposed to human linguistic variety.
4. **Cache Stampede / Thundering Herd**: What happens when multiple concurrent requests miss the cache simultaneously for the same expensive prompt.